# Training Example (COMP560)

Demonstrates training a ResNet50-based face recognition model using ArcFace or Triplet loss, and generating prediction CSVs.

Set `MODE = "train"` to train, `MODE = "predict"` to generate predictions from a checkpoint.

## Configuration

In [ ]:
MODE           = "train"   # "train" or "predict"

# Training
DATA_ROOT      = "./datasets/dataset_a"
SAVE_DIR       = "./checkpoints"
LOSS           = "arcface"   # "arcface" or "triplet"
LR             = 1e-4
WEIGHT_DECAY   = 1e-4
EPOCHS         = 20
WARMUP_EPOCHS  = 2
MARGIN         = 0.3
SAVE_EVERY     = 5

# Prediction
CHECKPOINT     = "./checkpoints/best_model.pth"
DATASET_ROOT   = "./datasets/dataset_a"
OUTPUT         = "predictions/dataset_a.csv"

# Shared
EMBEDDING_DIM  = 512
BATCH_SIZE     = 64
IMAGE_SIZE     = 112
NUM_WORKERS  = 0  # >0 causes multiprocessing errors in Jupyter on macOS

import torch
# Auto-detects: CUDA > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

In [ ]:
import os
import math
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision import models, transforms
from PIL import Image
from tqdm import tqdm

## Dataset

In [ ]:
def load_required_parquet(root, parquet_file, required_columns):
    parquet_path = os.path.join(root, parquet_file)
    if not os.path.exists(parquet_path):
        raise FileNotFoundError(f"Missing parquet file: {parquet_path}")

    df = pd.read_parquet(parquet_path)
    missing = [col for col in required_columns if col not in df.columns]
    if missing:
        raise ValueError(f"{parquet_file} is missing required columns: {missing}")
    return df


def resolve_image_path(root, image_path):
    rel = str(image_path).replace("\\", "/").lstrip("./")
    candidates = [
        os.path.join(root, rel),
        os.path.join(root, "images", rel),
        os.path.join(root, "images", os.path.basename(rel)),
    ]
    for candidate in candidates:
        if os.path.exists(candidate):
            return candidate
    return candidates[0]


class FaceTrainDataset(Dataset):
    def __init__(self, root, parquet_file="test.parquet", image_size=(112, 112)):
        self.root = root
        df = load_required_parquet(root, parquet_file, ["image_path", "template_id"])
        unique_templates = sorted(df["template_id"].unique())
        self.tid_to_label = {tid: i for i, tid in enumerate(unique_templates)}
        self.num_classes = len(unique_templates)
        self.image_paths = df["image_path"].tolist()
        self.labels = [self.tid_to_label[tid] for tid in df["template_id"].values]
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.RandomHorizontalFlip(),
            transforms.RandomRotation(10),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.1),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = resolve_image_path(self.root, self.image_paths[idx])
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image not found: {path}. Check image_path values in parquet and dataset root.")
        image = Image.open(path).convert("RGB")
        return self.transform(image), self.labels[idx]


class FaceTestDataset(Dataset):
    def __init__(self, root, image_size=(112, 112)):
        self.root = root
        df = load_required_parquet(root, "test.parquet", ["image_path", "template_id", "media_id"])
        self.image_paths = df["image_path"].tolist()
        self.template_ids = df["template_id"].values
        self.media_ids = df["media_id"].values
        self.transform = transforms.Compose([
            transforms.Resize(image_size),
            transforms.ToTensor(),
            transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ])

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        path = resolve_image_path(self.root, self.image_paths[idx])
        if not os.path.exists(path):
            raise FileNotFoundError(f"Image not found: {path}. Check image_path values in parquet and dataset root.")
        image = Image.open(path).convert("RGB")
        return self.transform(image), idx

## Loss Functions

In [ ]:
class ArcFaceLoss(nn.Module):
    def __init__(self, embedding_dim, num_classes, s=30.0, m=0.50):
        super().__init__()
        self.s = s
        self.m = m
        self.weight = nn.Parameter(torch.FloatTensor(num_classes, embedding_dim))
        nn.init.xavier_uniform_(self.weight)

    def forward(self, embeddings, labels):
        cosine = F.linear(F.normalize(embeddings), F.normalize(self.weight))
        theta = torch.acos(torch.clamp(cosine, -1.0 + 1e-7, 1.0 - 1e-7))
        one_hot = torch.zeros_like(cosine).scatter_(1, labels.unsqueeze(1), 1.0)
        target_logits = torch.cos(theta + self.m * one_hot)
        return F.cross_entropy(target_logits * self.s, labels)


class TripletLoss(nn.Module):
    def __init__(self, margin=0.3):
        super().__init__()
        self.margin = margin

    def forward(self, embeddings, labels):
        embeddings = F.normalize(embeddings, p=2, dim=1)
        dist_mat = 1 - torch.mm(embeddings, embeddings.t())
        labels = labels.unsqueeze(0)
        same_identity = labels == labels.t()
        loss = torch.tensor(0.0, device=embeddings.device)
        count = 0
        for i in range(embeddings.size(0)):
            pos_mask = same_identity[i].clone()
            pos_mask[i] = False
            neg_mask = ~same_identity[i]
            if pos_mask.any() and neg_mask.any():
                hardest_pos = dist_mat[i][pos_mask].max()
                hardest_neg = dist_mat[i][neg_mask].min()
                loss += F.relu(hardest_pos - hardest_neg + self.margin)
                count += 1
        return loss / max(count, 1)

## Model

In [ ]:
class TrainableModel(nn.Module):
    def __init__(self, embedding_dim=512):
        super().__init__()
        self.backbone = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V2)
        self.backbone.fc = nn.Linear(2048, embedding_dim)
        self._embedding_dim = embedding_dim

    @property
    def embedding_dim(self):
        return self._embedding_dim

    def forward(self, images):
        return self.backbone(images)

    def encode(self, images):
        return F.normalize(self.forward(images), p=2, dim=1)

## Template Aggregation

In [ ]:
def aggregate_template_features(embeddings, template_ids, media_ids):
    template_features = {}
    for tid in np.unique(template_ids):
        mask = template_ids == tid
        t_embs = embeddings[mask]
        t_mids = media_ids[mask]
        media_feats = []
        for mid in np.unique(t_mids):
            media_feats.append(t_embs[t_mids == mid].mean(axis=0))
        feat = np.sum(media_feats, axis=0)
        feat = feat / (np.linalg.norm(feat) + 1e-8)
        template_features[tid] = feat
    return template_features

## Training Loop

In [ ]:
def train(data_root, save_dir, loss_name, lr, weight_decay, epochs, warmup_epochs,
          margin, save_every, embedding_dim, batch_size, image_size, num_workers, device):
    device = torch.device(device if (torch.cuda.is_available() or (device == "mps" and torch.backends.mps.is_available())) else "cpu")
    dataset = FaceTrainDataset(data_root, image_size=(image_size, image_size))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=True,
                            num_workers=num_workers, pin_memory=(device.type == 'cuda'), drop_last=True)
    print(f"Training set: {len(dataset)} images, {dataset.num_classes} identities")

    model = TrainableModel(embedding_dim=embedding_dim).to(device)
    if loss_name == "arcface":
        criterion = ArcFaceLoss(embedding_dim, dataset.num_classes).to(device)
    elif loss_name == "triplet":
        criterion = TripletLoss(margin=margin)
    else:
        raise ValueError(f"Unknown loss: {loss_name}")

    params = list(model.parameters())
    if hasattr(criterion, 'parameters'):
        params += list(criterion.parameters())
    optimizer = torch.optim.AdamW(params, lr=lr, weight_decay=weight_decay)

    total_steps = len(dataloader) * epochs
    warmup_steps = len(dataloader) * warmup_epochs
    def lr_lambda(step):
        if step < warmup_steps:
            return step / max(warmup_steps, 1)
        progress = (step - warmup_steps) / max(total_steps - warmup_steps, 1)
        return 0.5 * (1 + math.cos(math.pi * progress))
    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda)

    save_path = Path(save_dir)
    save_path.mkdir(parents=True, exist_ok=True)
    best_loss = float('inf')

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        pbar = tqdm(dataloader, desc=f"Epoch {epoch+1}/{epochs}")
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            loss = criterion(model(images), labels)
            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            scheduler.step()
            epoch_loss += loss.item()
            pbar.set_postfix(loss=f"{loss.item():.4f}", lr=f"{scheduler.get_last_lr()[0]:.6f}")

        avg_loss = epoch_loss / len(dataloader)
        print(f"Epoch {epoch+1}: avg_loss={avg_loss:.4f}")

        if avg_loss < best_loss:
            best_loss = avg_loss
            torch.save({
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss, 'embedding_dim': embedding_dim,
            }, save_path / "best_model.pth")
            print(f"  Saved best model (loss={avg_loss:.4f})")

        if (epoch + 1) % save_every == 0:
            torch.save({
                'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'loss': avg_loss, 'embedding_dim': embedding_dim,
            }, save_path / f"checkpoint_epoch{epoch+1}.pth")

    print(f"\nTraining complete. Best loss: {best_loss:.4f}")
    print(f"Checkpoints saved to: {save_dir}")

## Prediction Generation

In [ ]:
def predict(checkpoint, dataset_root, output, embedding_dim, batch_size, image_size, num_workers, device):
    device = torch.device(device if (torch.cuda.is_available() or (device == "mps" and torch.backends.mps.is_available())) else "cpu")
    model = TrainableModel(embedding_dim=embedding_dim).to(device)
    ckpt = torch.load(checkpoint, map_location=device, weights_only=True)
    model.load_state_dict(ckpt["model_state_dict"])
    model.eval()
    print(f"Loaded checkpoint: {checkpoint} (epoch {ckpt.get('epoch', '?')})")

    dataset = FaceTestDataset(dataset_root, image_size=(image_size, image_size))
    dataloader = DataLoader(dataset, batch_size=batch_size, shuffle=False,
                            num_workers=num_workers, pin_memory=(device.type == 'cuda'))
    pairs_df = load_required_parquet(dataset_root, "pairs.parquet", ["template_id_1", "template_id_2"])
    print(f"Images: {len(dataset)}, Pairs: {len(pairs_df)}")

    emb_list, idx_list = [], []
    with torch.inference_mode():
        for images, indices in tqdm(dataloader, desc="Encoding"):
            emb_list.append(model.encode(images.to(device)).cpu().numpy())
            idx_list.append(indices.numpy())

    embeddings = np.vstack(emb_list)
    indices = np.concatenate(idx_list)
    embeddings = embeddings[np.argsort(indices)]

    print("Aggregating template features...")
    template_features = aggregate_template_features(embeddings, dataset.template_ids, dataset.media_ids)

    print("Computing pair scores...")
    tid_list = sorted(template_features.keys())
    tid_to_idx = {tid: i for i, tid in enumerate(tid_list)}
    feat_matrix = np.zeros((len(tid_list), embedding_dim), dtype=np.float32)
    for tid, feat in template_features.items():
        feat_matrix[tid_to_idx[tid]] = feat

    t1s = pairs_df["template_id_1"].values
    t2s = pairs_df["template_id_2"].values
    scores = np.zeros(len(pairs_df), dtype=np.float32)
    batch = 500_000
    for i in tqdm(range(0, len(pairs_df), batch), desc="Scoring"):
        end = min(i + batch, len(pairs_df))
        idx1 = np.array([tid_to_idx[t] for t in t1s[i:end]])
        idx2 = np.array([tid_to_idx[t] for t in t2s[i:end]])
        scores[i:end] = np.sum(feat_matrix[idx1] * feat_matrix[idx2], axis=1)

    Path(output).parent.mkdir(parents=True, exist_ok=True)
    pd.DataFrame({"template_id_1": t1s, "template_id_2": t2s, "score": scores}).to_csv(output, index=False)
    print(f"Predictions saved to: {output} ({len(scores)} pairs)")

## Run

In [ ]:
if MODE == "train":
    train(
        data_root=DATA_ROOT, save_dir=SAVE_DIR, loss_name=LOSS,
        lr=LR, weight_decay=WEIGHT_DECAY, epochs=EPOCHS,
        warmup_epochs=WARMUP_EPOCHS, margin=MARGIN, save_every=SAVE_EVERY,
        embedding_dim=EMBEDDING_DIM, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE,
        num_workers=NUM_WORKERS, device=DEVICE,
    )
elif MODE == "predict":
    predict(
        checkpoint=CHECKPOINT, dataset_root=DATASET_ROOT, output=OUTPUT,
        embedding_dim=EMBEDDING_DIM, batch_size=BATCH_SIZE, image_size=IMAGE_SIZE,
        num_workers=NUM_WORKERS, device=DEVICE,
    )
else:
    raise ValueError(f"Unknown MODE: {MODE!r}. Set MODE = 'train' or 'predict'.")